In [3]:
import kagglehub
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, confusion_matrix
import shap


Load the Data

In [4]:
path = kagglehub.dataset_download("redwankarimsony/heart-disease-data")
data_csv_path = path + r"\heart_disease_uci.csv"
df = pd.read_csv(data_csv_path)

In [5]:
df.head()
df['num'] = (df['num'] > 0).astype(int)

In [16]:
X = df.drop(columns=["num","id"])
y = df["num"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

categorical_cols = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'thal']
numeric_cols = ['age', 'trestbps', 'chol', 'thalch', 'oldpeak', 'ca']

In [17]:
categorical_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

numeric_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

preprocessor = ColumnTransformer([
    ('cat', categorical_pipe, categorical_cols),
    ('num', numeric_pipe, numeric_cols)
])

In [18]:
X_train_preprocessed = preprocessor.fit_transform(X_train)
X_test_preprocessed = preprocessor.transform(X_test)

In [ ]:
# Train the model
model = XGBClassifier(n_estimators=300, random_state=42, eval_metric="logloss")
model.fit(X_train_preprocessed, y_train)

# Predictions
preds = model.predict(X_test_preprocessed)
probs = model.predict_proba(X_test_preprocessed)[:, 1]

# Evaluation
print("Accuracy:", accuracy_score(y_test, preds))
print(classification_report(y_test, preds))
print("ROC-AUC:", roc_auc_score(y_test, probs))
print(confusion_matrix(y_test, preds))

# SHAP plot
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test_preprocessed)
shap.summary_plot(shap_values, X_test_preprocessed, feature_names=preprocessor.get_feature_names_out())